# 01 · Синтез данных и таймстампы

Из скриптовых диалогов (`data/gold/*.json`) синтезируем аудиозвонки и перегенерируем таймстампы **по реальному синтезу**.

Пайплайн:

```
gold/<id>.json (текст, разметка)
   -> Silero TTS v4_ru  (operator=xenia, client=aidar, 48 кГц)
   -> склейка реплик с паузами 0.35 c  -> data/audio/<id>_orig.wav
   -> телефонный канал (EQ, 8 кГц, G.711 mu-law) -> data/audio/<id>.wav
   -> обновить dialogue[...].{start,end} и commitment.audio_{start,end} в gold
```

Аудио и venv в git не попадают (см. `research/.gitignore`). В репо — только текстовая разметка с реальными таймстампами.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd()
assert (PROJECT / "data" / "gold").exists(), "Запускай из services/api/research"

import synthesize_calls as sc

In [ ]:
# Первый запуск скачает модель Silero (torch.hub, ~150 МБ), дальше — кэш.
# Если _orig.wav уже совпадает по длительности с gold — пересинтез не нужен, обновится только phone-версия.
sc.main()

In [ ]:
import json

d = json.load(open("data/gold/explicit_deadline_01.json", encoding="utf-8"))
for t in d["dialogue"]:
    print(f"  [{t['start']:6.2f}-{t['end']:6.2f}] {t['speaker']:8s} {t['text'][:55]}")
for c in d["expected_commitments"]:
    print(f"  коммент [{c['audio_start']:.2f}-{c['audio_end']:.2f}] \"{c['context_quote'][:45]}\"")
print("\nПроверка равенства конца последней реплики длине аудио:")
orig = d["dialogue"][-1]["end"]
print(f"  last end = {orig:.2f} s")

## Результат

- `data/gold/*.json` — таймстампы перегенерированы по реальному синтезу.
- `data/audio/<id>.wav` — «телефонный» канал (16 кГц, mono) для T-one / pyannote.
- `data/audio/<id>_orig.wav` — исходный 48 кГц (для референса).

Дальше: `02_asr_tone` — ASR через T-one (T-Банк) + диаризация pyannote.

Полезные команды:

```bash
cd services/api/research
uv run python synthesize_calls.py        # перегенерировать всё
uv run ruff check .                       # линт
uv run mypy schemas.py generate_data.py synth_tts.py synthesize_calls.py
```